In [ ]:
from mailjet_rest import Client
import os
from dotenv import load_dotenv
from agents import function_tool, Agent, WebSearchTool, ModelSettings, Runner, trace, AsyncOpenAI, OpenAIResponsesModel
from pydantic import BaseModel, Field
from IPython.display import Markdown, display




In [ ]:
load_dotenv(override=True)

In [ ]:
api_key = os.environ["MJ_APIKEY_PUBLIC"]
api_secret = os.environ["MJ_APIKEY_PRIVATE"]
sender_email = os.environ["SENDER_EMAIL"]
sender_name = os.environ["SENDER_NAME"]

In [ ]:
email = os.getenv('RECEIVER_EMAIL')
print(email)

@function_tool
def send_email(subject: str, text_part: str = None, html_part: str = None ):
    mailjet = Client(auth=(api_key, api_secret), version="v3.1")

    data = {
        "Messages": [{
            "From": { "Email": sender_email, "Name": 'AI News Agent' },
            "To": [ {"Email": email} ],
            "Subject": subject,
            "TextPart": text_part or "",
            "HTMLPart": html_part or "",
        }]
    }

    result = mailjet.send.create(data=data)

    print(result.status_code)
    print(result.json())


In [ ]:
send_email( subject="Test Subject", html_part="<h3>This is a test email.</h3>")

In [ ]:
instructions = f"""You are a web search agent for NewReplicaAI.

Your task is to find and summarize the latest technology news in the following areas:
- Artificial Intelligence (AI)
- Software Development
- Smartphone Reviews
- Gadgets

For each category, identify one relevant and recent topic (from the current year only) and produce a concise summary.

Guidelines:
- Each summary must not exceed 300 words.
- Focus only on relevant and recent developments; ignore unrelated information.
- Do not include opinions, assumptions, or speculation.
- Base your summary strictly on verified information from your search results.
- Clearly include the title of the news article or source.
- Always cite the source of the information.
- Write in a clear, simple, and easy-to-understand manner.
- Ensure the summary is objective, factual, and unbiased.
- Organize the output in a structured and readable format.

- you have a tool for sending email after the infomarting is gathered and well structured, create a 
    email with the subject "Latest Technology News Summary" and include the summaries in the email body as HTML
"""


In [ ]:
openai_api_key = os.getenv('OPENAI_API_KEY')
print(openai_api_key)

agent_client = AsyncOpenAI(api_key=openai_api_key)
agent_model = OpenAIResponsesModel(model="gpt-4o-mini", openai_client=agent_client)

In [ ]:
class OutputSchema(BaseModel):
    title: str = Field(description="The title of the news article or source.")
    summary: str = Field(description="A concise summary of the latest news in AI, Software development, Smart Phone reviews and Gadgets based on the web search results.")
    
class SearchAgentOutput(BaseModel):
    results: list[OutputSchema] = Field(description="A list of summaries for each category based on the web search results.")

In [ ]:
search_agent = Agent(
    name="Search Agent", 
    instructions=instructions,
    model=agent_model,
    tools=[WebSearchTool(search_context_size='low'), send_email],
    model_settings=ModelSettings(temperature=0, tool_choice='required'),
    output_type=SearchAgentOutput
)

In [ ]:
message = "What are the latest news in AI, Software development, Smart Phone reviews and Gadgets?"
with trace("AI Search"):
    runner = await Runner.run(search_agent, message)
    print([display(Markdown(f"**Title:** {item.title}\n\n**Summary:** {item.summary}") for item in runner.final_output.results)])